In [1]:
import os
import shutil
import pickle

### Functions

### Constants

In [2]:
# project
try:
    str_project = os.getcwd().split('/')[4].replace('_','-')
except IndexError:
    str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')

Project: 20241112-simple-model-test


### Create ```app``` directory

In [3]:
str_dirname = 'app'
try:
    os.mkdir(str_dirname)
except FileExistsError:
    pass

### Place files appropriately

### Write ```requirements.txt``` to ```app/``` directory

In [4]:
%%writefile app/requirements.txt

waitress==2.1.1
flask==2.3.2

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm
pandas
scikit_learn
boto3
catboost
xgboost
optbinning
xmltodict

Overwriting app/requirements.txt


### Write ```app.py``` to ```app/``` directory

In [5]:
%%writefile app/app.py

from flask import Flask, request, jsonify, Response
import pickle
import traceback
import logging
from waitress import serve
import pandas as pd
import json
pd.options.mode.chained_assignment = None # suppress warning

# set up logging
logging.basicConfig(
    filename='flask_app.log', 
    level=logging.DEBUG, 
    format='%(asctime)s %(levelname)s %(message)s',
)

# instantiate app
app = Flask(__name__)

# route the model to http://127.0.0.1:5000/
@app.route('/', methods=['GET','POST']) # GET for status code, POST for predictions
# logic for GET and POST requests
def predict():
    if request.method == 'GET':
        # log it
        logging.info('GET request received')
        # get status code
        int_status_code = Response(status=200).status_code
        # return
        return f'Status code: {int_status_code}'
    elif request.method == 'POST':
        try:
            # import parser
            str_message = 'Loading parser...'
            logging.info(str_message)
            print(str_message)
            print('')
            cls_parser = pickle.load(open('cls_parser.pkl', 'rb'))
            
            # get payload
            str_message = 'Getting request...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_json_request = request.get_json()
            str_json_request = json.dumps(dict_json_request)
            
            # parse payload
            str_message = 'Parsing payload...'
            logging.info(str_message)
            print(str_message)
            cls_parser.get_data(str_request=str_json_request)
            cls_parser.engineer_pmt_hx()
            cls_parser.preprocessing()
            cls_parser.get_predictions()
            cls_parser.interpolate()
            cls_parser.adverse_action()
            cls_parser.counter_offers()
            cls_parser.generate_response()
            # extract output
            str_message = 'Extracting output...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_response = cls_parser.dict_response
            # return dict_response
            return dict_response
        except Exception as e:
            str_message = 'Exception occurred'
            logging.error(
                str_message, 
                exc_info=True,
            )
            #return traceback in json
            return jsonify({'error': str(e)})

# run app
if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)

Overwriting app/app.py


### Copy ```cls_parser.pkl``` to```app/```

In [6]:
str_filename = 'cls_parser.pkl'
str_source = f'../06_parser/output/{str_filename}'
str_destination = f'./app/{str_filename}'
shutil.copyfile(str_source, str_destination)

'./app/cls_parser.pkl'

### Copy local code files

In [7]:
list_str_filenames = [
    'api.py',
    'preprocessing.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
]
for str_filename in list_str_filenames:
    str_source = f'../06_parser/{str_filename}'
    str_destination = f'./app/{str_filename}'
    shutil.copyfile(str_source, str_destination)